# Stage 3 | Master dataset (backW5, forW2)

Fast build of the master parquet using minimal TA features (RSI, UO, Bollinger, z-scores, EMA crosses, time, basic patterns).


In [ ]:
# Activate local venv: source .venv/bin/activate
# %pip install pandas pandas_ta pyarrow pyyaml tqdm


In [21]:
from __future__ import annotations

import gc
from pathlib import Path
from typing import List, Set, Tuple

import pandas as pd
import pandas_ta as ta
import yaml
from tqdm.notebook import tqdm

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
paths_cfg = yaml.safe_load((PROJECT_ROOT / 'configs' / 'paths_stage2.yaml').read_text())

DATASETS_DIR = (PROJECT_ROOT / paths_cfg['datasets_dir']).resolve()
PROCESSED_DIR = (PROJECT_ROOT / paths_cfg['processed_dir']).resolve()
OUTPUT_DIR = DATASETS_DIR.parent / 'masters'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_BACK_W = 5
TARGET_FOR_W = 2
OUTPUT_FILE = OUTPUT_DIR / f"master_dataset_b{TARGET_BACK_W}_f{TARGET_FOR_W}.parquet"

label_path = PROJECT_ROOT / 'stage1' / 'labeled_symbols.csv'
labeled_symbols = pd.read_csv(label_path)['symbol'].astype(str).str.strip().str.upper()
ALLOWED_SYMBOLS: Set[str] = list(set(labeled_symbols))[:1]

print(f'Project root: {PROJECT_ROOT}')
print(f'Datasets dir: {DATASETS_DIR}')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Output: {OUTPUT_FILE}')
print(f'Labeled symbols: {len(ALLOWED_SYMBOLS)}')


Project root: /home/kamil/binance-bot
Datasets dir: /home/kamil/binance-bot/stage1/stage2_data/datasets
Processed dir: /home/kamil/binance-bot/stage1/stage2_data/processed
Output: /home/kamil/binance-bot/stage1/stage2_data/masters/master_dataset_b5_f2.parquet
Labeled symbols: 1


In [22]:
def parse_symbol_interval(path: Path) -> Tuple[str, str]:
    parts = path.stem.split('_')
    if len(parts) < 2:
        raise ValueError(f'Unexpected filename: {path.name}')
    return parts[0].upper(), parts[1]


def load_price(symbol: str, interval: str) -> pd.DataFrame:
    df = pd.read_parquet(PROCESSED_DIR / f'{symbol}_{interval}.parquet')
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = 'ts'
    return df.sort_index()


def load_labels(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df['ts'] = pd.to_datetime(df['ts'], utc=True)
    return df.set_index('ts').sort_index()


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['RSI'] = df.ta.rsi(length=14)
    df['ULTOSC'] = df.ta.uo()
    df['pct_change'] = df['close'].pct_change()

    bb = df.ta.bbands(length=20, std=2)
    if bb is not None:
        df = pd.concat([df, bb], axis=1)

    df['zscore_price'] = df.ta.zscore(close=df['close'], length=30)
    df['zscore_vol'] = df.ta.zscore(close=df['volume'], length=30)

    df['ema_1'] = df['close']
    df['ema_20'] = df.ta.ema(length=20)
    df['ema_50'] = df.ta.ema(length=50)
    df['ema_100'] = df.ta.ema(length=100)

    df['cross_1_20'] = df['ema_1'] - df['ema_20']
    df['cross_20_50'] = df['ema_20'] - df['ema_50']
    df['cross_50_100'] = df['ema_50'] - df['ema_100']
    df['cross_1_50'] = df['ema_1'] - df['ema_50']

    df['month'] = df.index.month
    df['day_of_week'] = df.index.dayofweek
    df['hour'] = df.index.hour

    patterns_list = ['doji', 'engulfing', 'hammer', 'shootingstar', 'morningstar', 'eveningstar']
    patterns = df.ta.cdl_pattern(name=patterns_list)
    if patterns is not None:
        df = pd.concat([df, patterns], axis=1)

    df.dropna(inplace=True)
    return df


def balance_and_encode(df: pd.DataFrame) -> pd.DataFrame:
    label_map = {'HOLD': 0, 'BUY': 1, 'SELL': 2}
    labeled = df.copy()
    labeled['label_id'] = labeled['label'].map(label_map)

    buy = labeled[labeled['label'] == 'BUY']
    sell = labeled[labeled['label'] == 'SELL']
    hold = labeled[labeled['label'] == 'HOLD']

    min_len = min(len(buy), len(sell))
    if min_len == 0:
        return pd.DataFrame()

    buy_s = buy.sample(n=min_len, random_state=42)
    sell_s = sell.sample(n=min_len, random_state=42)
    hold_s = hold.sample(n=min_len, random_state=42)

    return pd.concat([buy_s, sell_s, hold_s]).sort_index()


In [23]:
pattern = f'*_backW{TARGET_BACK_W}_forW{TARGET_FOR_W}.parquet'
candidate_files = sorted(DATASETS_DIR.glob(pattern))
target_files = [p for p in candidate_files if parse_symbol_interval(p)[0] in ALLOWED_SYMBOLS]

print(f'Building master for backW={TARGET_BACK_W}, forW={TARGET_FOR_W}')
print(f'Found {len(target_files)} labeled pairs (out of {len(candidate_files)} files)')

processed_dfs: List[pd.DataFrame] = []
for file in tqdm(target_files, desc='files'):
    symbol, interval = parse_symbol_interval(file)
    try:
        labels = load_labels(file)
        price = load_price(symbol, interval)
        df_raw = price.join(labels, how='inner')
        if df_raw.empty:
            print(f'[!] No overlap for {file.name}')
            continue

        if df_raw['backW'].iloc[0] != TARGET_BACK_W or df_raw['forW'].iloc[0] != TARGET_FOR_W:
            continue

        df_feat = add_features(df_raw)
        if df_feat.empty:
            continue

        df_final = balance_and_encode(df_feat)
        if df_final.empty:
            continue

        cols_to_drop = ['ema_1', 'ema_20', 'ema_50', 'ema_100', 'open', 'high', 'low', 'close', 'volume']
        df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], errors='ignore')
        df_final = df_final.reset_index().rename(columns={'index': 'ts'})
        df_final['symbol'] = symbol
        processed_dfs.append(df_final)
    except Exception as exc:
        print(f'Error {file.name}: {exc}')
    finally:
        gc.collect()

if processed_dfs:
    master_df = pd.concat(processed_dfs, ignore_index=True)
    master_df = master_df.sort_values('ts').reset_index(drop=True)
    master_df.to_parquet(OUTPUT_FILE)
    print(f'Done. Saved {len(master_df):,} rows -> {OUTPUT_FILE}')
else:
    master_df = pd.DataFrame()
    print('No data available for the requested window and labeled symbols.')


Building master for backW=5, forW=2
Found 1 labeled pairs (out of 436 files)


files:   0%|          | 0/1 [00:00<?, ?it/s]

[i] Requires TA-Lib to use engulfing. (pip install TA-Lib)
[i] Requires TA-Lib to use hammer. (pip install TA-Lib)
[i] Requires TA-Lib to use shootingstar. (pip install TA-Lib)
[i] Requires TA-Lib to use morningstar. (pip install TA-Lib)
[i] Requires TA-Lib to use eveningstar. (pip install TA-Lib)
Done. Saved 2,931 rows -> /home/kamil/binance-bot/stage1/stage2_data/masters/master_dataset_b5_f2.parquet


In [ ]:
master_df.head() if not master_df.empty else master_df
